# Student Dropout Prediction Notebook

This notebook reproduces your script as a step-by-step workflow:
1. Load libraries and dataset
2. Clean and preprocess target labels
3. Visualize correlations and class-wise distributions
4. Train and evaluate a Random Forest classifier
5. Compare split seeds and cross-validation results
6. Plot normalized confusion matrix
7. Retrain using only 1% of training data

In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

Question 1

In [ ]:
# Load the UCI dropout dataset (semicolon-separated)
df = pd.read_csv("data.csv", sep=";")

print("Dataset shape:", df.shape)
display(df.head())

print("nb de dropout : " + str(df[df['Target'] == "Dropout"].shape))

print("nb de enrolled : " +str(df[df['Target'] == "Enrolled"].shape))

print("nb de graduate : " + str(df[df['Target'] == "Graduate"].shape))

# Map Target to binary: 0=Dropout, 1=Enrolled/Graduate
target_mapping = {'Dropout': 0, 'Enrolled': 1, 'Graduate': 1}
df['Target'] = df['Target'].map(target_mapping)

# Drop potential NaNs after mapping
df = df.dropna(subset=['Target'])

X = df.drop('Target', axis=1)
y = df['Target']

print(f'Dataset shape: {df.shape}')
print(y.value_counts().rename(index={0: 'Dropout (0)', 1: 'Success (1)'}))

Question 2

In [ ]:
# Correlation matrix to detect redundant features
plt.figure(figsize=(12, 10))
corr_matrix = X.corr()
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Matrix (Check for Redundancy)')
plt.tight_layout()
plt.show()

In [ ]:
# Normalize features for distribution visualization
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_scaled['Target'] = y.reset_index(drop=True)

# Plot a subset of features to avoid visual clutter
features_to_plot = X.columns[:10]
X_melted = pd.melt(X_scaled, id_vars=['Target'], value_vars=features_to_plot)

plt.figure(figsize=(12, 6))
sns.boxplot(x='variable', y='value', hue='Target', data=X_melted)
plt.title('Normalized Feature Distributions by Class')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

X_scaled = X_scaled.drop('Target', axis=1)

## 2) Train/Test Split and Baseline Classifier